In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import os
import re
import torch.utils.data as data

In [2]:
class TextDataset(data.Dataset):
    def __init__(self, path, seq_length, step=3):
        self.seq_length = seq_length
        self.step = step
        
        with open(path, 'r', encoding='utf-8') as f:
            names = [line.strip().lower() for line in f.readlines()]
            text = '\n'.join(names)
        text = text.replace('\ufeff', '')
        

        self.chars = sorted(list(set(text)))
        self.char_to_int = {c: i for i, c in enumerate(self.chars)}
        self.int_to_char = {i: c for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars) 

        self.encoded_text = torch.tensor([self.char_to_int[c] for c in text], dtype=torch.long)

    def __len__(self):
        return (len(self.encoded_text) - self.seq_length) // self.step

    def __getitem__(self, idx):
        start_idx = idx * self.step
        
        chunk = self.encoded_text[start_idx : start_idx + self.seq_length + 1]
        
        input_seq = chunk[:-1] 
        target_seq = chunk[1:]
        
        return input_seq, target_seq

In [4]:
d_train = TextDataset('dinos.txt', 64, step=1)
n = len(d_train)
train_size = int(n*0.8)
val_size = int(n*0.1)
teset_size = int(n*0.1)
train_indices = range(0, train_size)
val_indices = range(train_size, train_size + val_size)
test_indices = range(train_size + val_size, n)
train_ds = data.Subset(d_train, train_indices)
val_ds = data.Subset(d_train, val_indices)
test_ds = data.Subset(d_train, test_indices)
train_loader = data.DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)
val_loader = data.DataLoader(val_ds, batch_size=256, shuffle=False, drop_last=True)
test_loader = data.DataLoader(test_ds, batch_size=256, shuffle=False, drop_last=True)

In [5]:
import torch
import torch.nn as nn

class CharGRU(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, n_layers=2):
        super(CharGRU, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        
        self.gru = nn.GRU(
            input_size=emb_dim, 
            hidden_size=hidden_dim, 
            num_layers=n_layers, 
            batch_first=True,
            dropout=0.2 
        )
        
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim

    def forward(self, x, hidden):

        embedded = self.embedding(x)

        out, hidden = self.gru(embedded, hidden)
        
        prediction = self.fc(out)
        
        return prediction, hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)

In [6]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print(device)
# Гиперпараметры
VOCAB_SIZE = len(d_train.chars) 
EMB_DIM = 64
HIDDEN_DIM = 256
N_LAYERS = 2
BATCH_SIZE = 256
LR = 0.001       
EPOCHS = 6   

mps


In [7]:
import torch.nn.functional as F

def generate_dino(model, start_char='A', temperature=0.8):
    model.eval() 

    input_idx = torch.tensor([[d_train.char_to_int[start_char]]]).to(device)
    
    hidden = model.init_hidden(1, device)
    
    generated_name = start_char
    

    while True:
        output, hidden = model(input_idx, hidden)
        
        logits = output[0, 0]

        probs = F.softmax(logits / temperature, dim=0)
        
        next_char_idx = torch.multinomial(probs, num_samples=1).item()
        next_char = d_train.int_to_char[next_char_idx]
        

        if next_char == '\n':
            break 

        generated_name += next_char
        input_idx = torch.tensor([[next_char_idx]]).to(device)
        
    return generated_name

In [8]:
model = CharGRU(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, N_LAYERS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


for epoch in range(EPOCHS):
    model.train() 
    train_loss = 0
    loop = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    for i, (inputs, targets) in enumerate(loop):

        inputs, targets = inputs.to(device), targets.to(device)
        

        hidden = model.init_hidden(BATCH_SIZE, device)
        

        optimizer.zero_grad()
        
        output, hidden = model(inputs, hidden)
        
        loss = criterion(output.view(-1, VOCAB_SIZE), targets.view(-1))
        
        loss.backward()
        

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)

        optimizer.step()
        
        train_loss += loss.item()
    avg_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}')
    print("Пример:", generate_dino(model, start_char='a', temperature=0.8))
    print("-" * 50)

Epoch 1/6: 100%|██████████| 62/62 [00:04<00:00, 12.69it/s]


Epoch 1/6 | Loss: 2.2019
Пример: a
--------------------------------------------------


Epoch 2/6: 100%|██████████| 62/62 [00:04<00:00, 13.86it/s]


Epoch 2/6 | Loss: 1.6296
Пример: anitonyx
--------------------------------------------------


Epoch 3/6: 100%|██████████| 62/62 [00:04<00:00, 13.83it/s]


...

Epoch 4/6 | Loss: 1.0917
Пример: antosaurus
--------------------------------------------------


Epoch 5/6: 100%|██████████| 62/62 [00:04<00:00, 13.64it/s]


Epoch 5/6 | Loss: 0.7893
Пример: alovenator
--------------------------------------------------


Epoch 6/6: 100%|██████████| 62/62 [00:04<00:00, 14.09it/s]

Epoch 6/6 | Loss: 0.5532
Пример: aurus
--------------------------------------------------


In [9]:
print(generate_dino(model, start_char='t', temperature=0.8)) 
print(generate_dino(model, start_char='t', temperature=1.2)) 
print(generate_dino(model, start_char='t', temperature=0.5)) 

toraptor
tosaurus
tor


In [10]:
import string
for letter in string.ascii_lowercase:
    while True:
        name = generate_dino(model, start_char=letter, temperature=0.8)
        if len(name) > 4: 
            print(f"{letter}: {name.capitalize()}")
            break

a: Aurus
b: Belosaurus
c: Chendosaurus
d: Dromeus
e: Entaraptor
f: Fruvenator
g: Gosaurus
h: Huenpelosaurus
i: Ilong
j: Jaurus
k: Kelta
l: Lophosaurus
m: Maiansaurus
n: Nosaurus
o: Oplosaurus
p: Pallewiangosaurus
q: Quanjiangosaurus
r: Ratops
s: Saurus
t: Tosaurus
u: Urinosaurus
v: Venator
w: Watlisaurus
x: Xianansaurus
y: Ycheiros
z: Zchuan
